# Azure ETL & Retrieval - RAG Pipeline Demo

This notebook walks through a complete Retrieval-Augmented Generation (RAG) pipeline — from raw documents to evaluated, guardrailed answers. Each section builds on the previous one, progressively adding layers of sophistication.

### What this notebook covers

| Section | Purpose |
|---|---|
| **1. ETL Pipeline** | Load documents, split them into chunks, embed, and index into a vector store |
| **2. Hybrid Search & Guardrails** | Run keyword + vector search with a similarity-based relevance guardrail |
| **3. RAG with DeepEval** | Generate answers and evaluate them with an LLM judge (Claude Haiku 4.5) |
| **4. StateGraph with Dynamic Routing** | A LangGraph state machine that retries with rephrased queries and dynamically adjusted parameters based on evaluation scores from LLM judge (Claude Haiku 4.5) |

### Environment toggle

Set `ENVIRONMENT=dev` for local testing (Chroma + OpenAI) or `ENVIRONMENT=prod` for full Azure (Blob Storage, Document Intelligence, Azure OpenAI, AI Search with Semantic Ranking).

## 1. ETL Pipeline

The pipeline follows four stages:

1. **Extract** — Load raw documents (PDF, Markdown, TXT) from local disk or Azure Blob Storage. PDFs are processed with Azure AI Document Intelligence for OCR.
2. **Chunk** — Split documents into overlapping text chunks using a recursive character splitter, preserving source metadata (filename, page, chunk index).
3. **Embed** — Convert each chunk into a vector using OpenAI embeddings (`text-embedding-3-small`).
4. **Index** — Store the vectors in a searchable index (Chroma in dev, Azure AI Search in prod) and update the document catalog for incremental ingest support.

In [1]:
# Setup and load environment
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")

print(f"ENVIRONMENT: {os.getenv('ENVIRONMENT', 'prod')}")

ENVIRONMENT: dev


### Extract: Load documents

The knowledge base is stored under `data/` with sub-folders for different document types (`manuals/`, `troubleshooting/`, `policies/`). The loader reads `.txt`, `.md`, and `.pdf` files, attaching metadata (source path, filename) that follows each chunk through the pipeline.

In [2]:
# Load documents (local for demo; use load_from_blob for Azure)
from src.extract import load_from_local, load_from_blob

# Option A: Local (no Azure required) - data/ is relative to project root
data_path = Path.cwd().parent / "data" if Path.cwd().name == "notebooks" else "data"
documents = load_from_local(str(data_path))
print(f"Loaded {len(documents)} documents")

# Option B: From Azure Blob (uncomment when AZURE_STORAGE_CONNECTION_STRING is set)
# documents = load_from_blob(
#     connection_string=os.getenv("AZURE_STORAGE_CONNECTION_STRING"),
#     container_name="documents",
# )

Loaded 108 documents


### Chunk: Split into retrievable segments

Documents are split into overlapping chunks using `RecursiveCharacterTextSplitter`. Overlap ensures that context at chunk boundaries isn't lost during retrieval. Each chunk inherits its parent document's metadata and receives a `chunk_index` for traceability back to its position in the original document.

In [3]:
# Chunk documents
from src.chunk import chunk_documents

chunks = chunk_documents(documents)
print(f"Created {len(chunks)} chunks")

Created 283 chunks


### Embed & Index: Full ingest pipeline

The `run_ingest` function orchestrates the full pipeline end-to-end: extract, chunk, embed, and upsert into the vector store. It also writes a document catalog (SQLite in dev, Azure SQL in prod) that tracks content hashes for incremental updates — re-running ingest only processes documents that have changed.

In [4]:
# Create index and run full ingest
# (Re-run this cell to refresh chunk indices if search results show "Chunk ?")
from src.ingest import run_ingest

# For local demo: use_local=True, ENVIRONMENT=dev
data_path = Path.cwd().parent / "data" if Path.cwd().name == "notebooks" else "data"
run_ingest(
    index_name="kb-index",
    use_local=True,
    local_path=str(data_path),
)

Loading from local data
Loaded 108 documents
Created 283 chunks
Uploaded 283 chunks to index
Updated catalog for 9 documents


## 2. Hybrid Search, RAG Answer & Relevance Guardrail

This section demonstrates three layers:

1. **Hybrid Search** — retrieve the top-n chunks and show their cosine similarity scores (1 = identical, 0 = orthogonal). This is the raw retrieval step.
2. **RAG Answer with Citations** — pass the retrieved chunks to the LLM to generate an answer that cites its sources using `[1]`, `[2]` markers.
3. **Guardrail Test** — repeat the search with a `score_threshold`. Scores are **cosine similarity** where:
   - **0.85–1.0**: Strongly related
   - **0.7–0.85**: Related
   - **0.5–0.7**: Weakly related
   - **< 0.5**: Likely irrelevant

   A threshold of **0.75** is a reasonable starting point. When every chunk's similarity falls below this, `NO_RELEVANT_CHUNKS` is returned and generation is skipped entirely.

In [5]:
from IPython.display import Markdown, display, clear_output
from src.search import hybrid_search, rag_retrieve_and_generate, NO_RELEVANT_CHUNKS
from src.index import get_vector_store
from src.embed import get_embeddings

embeddings = get_embeddings()
vector_store = get_vector_store(embeddings)
query = "What are the primary limitations of traditional VectorRAG when analyzing unstructured financial documents like earnings calls, and how can integrating Knowledge Graphs help?"

# --- Part 1: Retrieve chunks (unfiltered) ---
results = hybrid_search(query=query, vector_store=vector_store, top_n=3)

display_text = f"### 🔍 Query: *\"{query}\"*\n<hr>\n\n"

display_text += "#### 🔎 Retrieved Chunks (cosine similarity: 1 = identical)\n\n"
if results == NO_RELEVANT_CHUNKS:
    display_text += "> No chunks found.\n\n"
else:
    for i, r in enumerate(results, 1):
        source_label = r["metadata"].get("source_display") or r["metadata"].get("source") or r["metadata"].get("filename", "unknown")
        sim = r.get("score", "N/A")
        sim_str = f"{sim:.4f}" if isinstance(sim, float) else str(sim)
        preview = r["content"][:200].replace("\n", " ")
        display_text += f"**[{i}]** `{source_label}` — similarity: `{sim_str}`\n"
        display_text += f"> {preview}...\n\n"

# --- Part 2: Generate answer with citations ---
answer, chunks = rag_retrieve_and_generate(
    query=query, top_n=3, vector_store=vector_store,
)

display_text += "<br>\n\n#### 🤖 RAG Answer (with citations)\n\n"
display_text += f"> {answer}\n\n"

if isinstance(chunks, list) and chunks:
    display_text += "**📚 Citations:**\n\n"
    for i, c in enumerate(chunks, 1):
        meta = c.get("metadata", {})
        source = meta.get("source_display", "unknown")
        preview = c.get("content", "")[:150].replace("\n", " ")
        display_text += f"**[{i}]** `{source}`\n> {preview}...\n\n"

# --- Part 3: Guardrail demo (off-topic query) ---
off_topic = "What is the purpose of the BMO AI Engineer role?"
guarded = hybrid_search(query=off_topic, vector_store=vector_store, top_n=3, score_threshold=0.75)

display_text += "<br>\n\n#### 🛡️ Guardrail Test (`score_threshold=0.75`)\n"

if guarded == NO_RELEVANT_CHUNKS:
    display_text += """
<div style="background-color: #fee2e2; border-left: 6px solid #ef4444; padding: 15px; border-radius: 4px;">
    <h4 style="margin-top:0; margin-bottom: 8px; color: #b91c1c;">🛑 GUARDRAIL TRIGGERED</h4>
    <p style="margin-bottom:0; color: #991b1b;">
        <b>System Signal:</b> <code>NO_RELEVANT_CHUNKS</code><br>
        <b>Action Taken:</b> All chunks fell below the similarity threshold. Generation is skipped to prevent hallucination.
    </p>
</div>
"""
else:
    display_text += f"\n<span style='color:#10b981; font-weight:bold;'>✅ PASSED GUARDRAIL:</span> {len(guarded)} chunks within threshold.\n\n"
    for i, r in enumerate(guarded, 1):
        sim = r.get("score", "N/A")
        sim_str = f"{sim:.4f}" if isinstance(sim, float) else str(sim)
        meta = r["metadata"]
        display_text += f"**[{i}]** `{meta.get('source_display', 'unknown')}` — similarity: `{sim_str}`\n\n"

clear_output(wait=True)
display(Markdown(display_text))

### 🔍 Query: *"What are the primary limitations of traditional VectorRAG when analyzing unstructured financial documents like earnings calls, and how can integrating Knowledge Graphs help?"*
<hr>

#### 🔎 Retrieved Chunks (cosine similarity: 1 = identical)

**[1]** `manuals/HybridRAG.pdf (Chunk 1)` — similarity: `0.7336`
> HybridRAG: Integrating Knowledge Graphs and Vector Retrieval Augmented Generation for Efficient Information Extraction Bhaskarjit Sarmah bhaskarjit.sarmah@blackrock.com BlackRock, Inc. Gurugram, India...

**[2]** `manuals/HybridRAG.pdf (Chunk 1)` — similarity: `0.7332`
> HybridRAG: Integrating Knowledge Graphs and Vector Retrieval Augmented Generation for Efficient Information Extraction Bhaskarjit Sarmah bhaskarjit.sarmah@blackrock.com BlackRock, Inc. Gurugram, India...

**[3]** `manuals/HybridRAG.pdf (Chunk 2)` — similarity: `0.7327`
> HybridRAG which retrieves context from both vector database and KG outperforms both traditional VectorRAG and GraphRAG indi- vidually when evaluated at both the retrieval and generation stages in term...

<br>

#### 🤖 RAG Answer (with citations)

> The primary limitations of traditional VectorRAG when analyzing unstructured financial documents, such as earnings calls, include challenges related to domain-specific terminology, complex formats of the documents, and the inherent difficulties in extracting meaningful insights from unstructured data. These limitations can lead to inaccurate predictions, overlooked insights, and unreliable analysis, which ultimately hinder decision-making for financial analysts [1][3].

Integrating Knowledge Graphs (KGs) with VectorRAG, as proposed in the HybridRAG approach, can help address these limitations by enhancing the question-answer (Q&A) systems for information extraction. The combination of KGs and VectorRAG allows for improved retrieval accuracy and answer generation by leveraging the structured relationships and contextual information provided by KGs alongside the vector-based retrieval capabilities. This integration enables the system to generate more accurate and contextually relevant answers, thereby improving the overall effectiveness of information extraction from complex financial documents [1][3].

**📚 Citations:**

**[1]** `manuals/HybridRAG.pdf (Chunk 1)`
> HybridRAG: Integrating Knowledge Graphs and Vector Retrieval Augmented Generation for Efficient Information Extraction Bhaskarjit Sarmah bhaskarjit.sa...

**[2]** `manuals/HybridRAG.pdf (Chunk 1)`
> HybridRAG: Integrating Knowledge Graphs and Vector Retrieval Augmented Generation for Efficient Information Extraction Bhaskarjit Sarmah bhaskarjit.sa...

**[3]** `manuals/HybridRAG.pdf (Chunk 2)`
> HybridRAG which retrieves context from both vector database and KG outperforms both traditional VectorRAG and GraphRAG indi- vidually when evaluated a...

<br>

#### 🛡️ Guardrail Test (`score_threshold=0.75`)

<div style="background-color: #fee2e2; border-left: 6px solid #ef4444; padding: 15px; border-radius: 4px;">
    <h4 style="margin-top:0; margin-bottom: 8px; color: #b91c1c;">🛑 GUARDRAIL TRIGGERED</h4>
    <p style="margin-bottom:0; color: #991b1b;">
        <b>System Signal:</b> <code>NO_RELEVANT_CHUNKS</code><br>
        <b>Action Taken:</b> All chunks fell below the similarity threshold. Generation is skipped to prevent hallucination.
    </p>
</div>


### Another Example

In [6]:
from IPython.display import Markdown, display, clear_output
from src.search import hybrid_search, rag_retrieve_and_generate, NO_RELEVANT_CHUNKS
from src.index import get_vector_store
from src.embed import get_embeddings

embeddings = get_embeddings()
vector_store = get_vector_store(embeddings)
query = "What were the specific Q3 2025 revenue figures for the top 5 companies in the Nifty 50 index?"

# --- Part 1: Retrieve chunks (unfiltered) ---
results = hybrid_search(query=query, vector_store=vector_store, top_n=3)

display_text = f"### 🔍 Query: *\"{query}\"*\n<hr>\n\n"

display_text += "#### 🔎 Retrieved Chunks (cosine similarity: 1 = identical)\n\n"
if results == NO_RELEVANT_CHUNKS:
    display_text += "> No chunks found.\n\n"
else:
    for i, r in enumerate(results, 1):
        source_label = r["metadata"].get("source_display") or r["metadata"].get("source") or r["metadata"].get("filename", "unknown")
        sim = r.get("score", "N/A")
        sim_str = f"{sim:.4f}" if isinstance(sim, float) else str(sim)
        preview = r["content"][:200].replace("\n", " ")
        display_text += f"**[{i}]** `{source_label}` — similarity: `{sim_str}`\n"
        display_text += f"> {preview}...\n\n"

# --- Part 2: Generate answer with citations ---
answer, chunks = rag_retrieve_and_generate(
    query=query, top_n=3, vector_store=vector_store,
)

display_text += "<br>\n\n#### 🤖 RAG Answer (with citations)\n\n"
display_text += f"> {answer}\n\n"

if isinstance(chunks, list) and chunks:
    display_text += "**📚 Citations:**\n\n"
    for i, c in enumerate(chunks, 1):
        meta = c.get("metadata", {})
        source = meta.get("source_display", "unknown")
        preview = c.get("content", "")[:150].replace("\n", " ")
        display_text += f"**[{i}]** `{source}`\n> {preview}...\n\n"

# --- Part 3: Guardrail demo ---
guarded = hybrid_search(query=query, vector_store=vector_store, top_n=3, score_threshold=0.75)

display_text += "<br>\n\n#### 🛡️ Guardrail Test (`score_threshold=0.75`)\n"

if guarded == NO_RELEVANT_CHUNKS:
    display_text += """
<div style="background-color: #fee2e2; border-left: 6px solid #ef4444; padding: 15px; border-radius: 4px;">
    <h4 style="margin-top:0; margin-bottom: 8px; color: #b91c1c;">🛑 GUARDRAIL TRIGGERED</h4>
    <p style="margin-bottom:0; color: #991b1b;">
        <b>System Signal:</b> <code>NO_RELEVANT_CHUNKS</code><br>
        <b>Action Taken:</b> All chunks fell below the similarity threshold. Generation is skipped to prevent hallucination.
    </p>
</div>
"""
else:
    display_text += f"\n<span style='color:#10b981; font-weight:bold;'>✅ PASSED GUARDRAIL:</span> {len(guarded)} chunks within threshold.\n\n"
    for i, r in enumerate(guarded, 1):
        sim = r.get("score", "N/A")
        sim_str = f"{sim:.4f}" if isinstance(sim, float) else str(sim)
        meta = r["metadata"]
        display_text += f"**[{i}]** `{meta.get('source_display', 'unknown')}` — similarity: `{sim_str}`\n\n"

clear_output(wait=True)
display(Markdown(display_text))

### 🔍 Query: *"What were the specific Q3 2025 revenue figures for the top 5 companies in the Nifty 50 index?"*
<hr>

#### 🔎 Retrieved Chunks (cosine similarity: 1 = identical)

**[1]** `manuals/HybridRAG.pdf (Chunk 15)` — similarity: `0.4232`
> HybridRAG: Integrating Knowledge Graphs and Vector Retrieval Augmented Generation for Efficient Information Extraction documents but such that we finally have access to both the actual financial docum...

**[2]** `manuals/HybridRAG.pdf (Chunk 15)` — similarity: `0.4232`
> HybridRAG: Integrating Knowledge Graphs and Vector Retrieval Augmented Generation for Efficient Information Extraction documents but such that we finally have access to both the actual financial docum...

**[3]** `manuals/HybridRAG.pdf (Chunk 16)` — similarity: `0.3951`
> Energy - Coal, Materials, Information Technology, Construction, Diversified, Metals, Energy - Power and Chemicals providing a substantial and diverse foundation for our study. We start the data collec...

<br>

#### 🤖 RAG Answer (with citations)

> I do not have enough information to answer your question about the specific Q3 2025 revenue figures for the top 5 companies in the Nifty 50 index. The provided context focuses on earnings reports for Q1 of the financial year 2024 and does not include data for Q3 2025.

**📚 Citations:**

**[1]** `manuals/HybridRAG.pdf (Chunk 15)`
> HybridRAG: Integrating Knowledge Graphs and Vector Retrieval Augmented Generation for Efficient Information Extraction documents but such that we fina...

**[2]** `manuals/HybridRAG.pdf (Chunk 15)`
> HybridRAG: Integrating Knowledge Graphs and Vector Retrieval Augmented Generation for Efficient Information Extraction documents but such that we fina...

**[3]** `manuals/HybridRAG.pdf (Chunk 16)`
> Energy - Coal, Materials, Information Technology, Construction, Diversified, Metals, Energy - Power and Chemicals providing a substantial and diverse ...

<br>

#### 🛡️ Guardrail Test (`score_threshold=0.75`)

<div style="background-color: #fee2e2; border-left: 6px solid #ef4444; padding: 15px; border-radius: 4px;">
    <h4 style="margin-top:0; margin-bottom: 8px; color: #b91c1c;">🛑 GUARDRAIL TRIGGERED</h4>
    <p style="margin-bottom:0; color: #991b1b;">
        <b>System Signal:</b> <code>NO_RELEVANT_CHUNKS</code><br>
        <b>Action Taken:</b> All chunks fell below the similarity threshold. Generation is skipped to prevent hallucination.
    </p>
</div>


## 3. RAG with DeepEval Evaluation

Distance thresholds catch obviously irrelevant results, but they can't judge whether a *generated answer* actually addresses the user's question. This is where **DeepEval** comes in — it uses a judge LLM (Claude Haiku 4.5) to score the RAG output on two dimensions:

| Metric | What it measures |
|---|---|
| **Answer Relevancy** | Does the generated answer address the original question? |
| **Contextual Relevancy** | Are the retrieved chunks actually relevant to the question? |

Both metrics produce a score between 0 and 1. If either falls below the `eval_threshold` (default 0.5), the system flags the response as unreliable and returns feedback suggesting the user rephrase their question or add more documents to the knowledge base.

This is a single-pass evaluation — retrieve, generate, evaluate, done. The next section builds on this by adding retry logic.

In [7]:
from IPython.display import Markdown, display, clear_output
from src.search import query_with_evaluation
from src.index import get_vector_store
from src.embed import get_embeddings

# 1. Initialize dependencies
embeddings = get_embeddings()
vector_store = get_vector_store(embeddings)
query = "If an enterprise RAG system is experiencing a high rate of indirect prompt injections and fabricated content, what specific mitigation strategies and architectural changes should be implemented?"

# 2. Execute the evaluated query
result = query_with_evaluation(
    vector_store=vector_store,
    query=query,
    top_n=3,
    eval_threshold=0.5,
)

# 3. Format the output beautifully using Markdown and HTML
if result["passed"]:
    status_color = "#10b981" # Emerald Green
    status_icon = "✅"
    status_text = "PASSED EVALUATION"
else:
    status_color = "#ef4444" # Rose Red
    status_icon = "🛑"
    status_text = "FAILED EVALUATION (Abstention Guardrail Triggered)"

metrics = result.get("metrics", {})
ans_rel = metrics.get('answer_relevancy', 0.0)
ctx_rel = metrics.get('contextual_relevancy', 0.0)

# Construct the rich display output
display_text = f"""
### 🔍 Query: *"{query}"*
<br>

**Evaluation Status:** <span style="color:{status_color}; font-weight:bold;">{status_icon} {status_text}</span>

**Agent Response:**
> {result["answer"]}

<br>

**DeepEval Metrics:**
* **Answer Relevancy:** `{ans_rel}`
* **Contextual Relevancy:** `{ctx_rel}`
"""

if not result["passed"] and result.get("feedback"):
    display_text += f"\n\n**Automated Feedback:**\n_{result['feedback']}_"

chunks = result.get("chunks", [])
if chunks:
    display_text += "\n\n**📚 Citations:**\n\n"
    for i, c in enumerate(chunks, 1):
        meta = c.get("metadata", {})
        source = meta.get("source_display", "unknown")
        preview = c.get("content", "")[:150].replace("\n", " ")
        display_text += f"**[{i}]** `{source}`\n> {preview}...\n\n"

clear_output(wait=True)
display(Markdown(display_text))


### 🔍 Query: *"If an enterprise RAG system is experiencing a high rate of indirect prompt injections and fabricated content, what specific mitigation strategies and architectural changes should be implemented?"*
<br>

**Evaluation Status:** <span style="color:#10b981; font-weight:bold;">✅ PASSED EVALUATION</span>

**Agent Response:**
> To address the high rate of indirect prompt injections and fabricated content in an enterprise RAG system, the following mitigation strategies and architectural changes should be implemented:

1. **Input Validation (M5)**: Implement robust input validation mechanisms to detect and reject user prompts or documents that contain malicious content or instructions. This will help safeguard the RAG pipeline from harmful inputs.

2. **Reinforcement of System Instruction (M4)**: Enhance the generator specifications to ensure a more restrictive handling of prompts and data retrieval. This can help prevent the generator from executing unwanted behaviors due to manipulated inputs.

3. **Access Limitation (M3)**: Restrict user access to only certain documents or services. Establishing a robust user authentication system can mitigate risks associated with unauthorized access to sensitive information.

4. **Anonymization (M0)**: Employ anonymization techniques to irreversibly remove personal information from data before it is used in the system. This reduces the risk of data leakage and protects sensitive information.

5. **Pseudonymization (M1)**: Use pseudonymization to substitute sensitive information with pseudonyms, allowing the system to produce accurate responses while minimizing exposure of actual sensitive data.

6. **Evaluation (M6)**: Implement evaluation metrics to improve the factual accuracy and reliability of the RAG system. Regularly assess the outputs for inaccuracies and correct them to reduce the chances of fabricated content.

7. **Synthetic Data (M2)**: Consider using synthetic data to replace sensitive data with artificially generated equivalents. This can help the RAG system operate with a reduced risk of exposing confidential information.

By integrating these strategies and architectural changes, the enterprise RAG system can better mitigate the risks associated with indirect prompt injections and the generation of fabricated content.

<br>

**DeepEval Metrics:**
* **Answer Relevancy:** `1.0`
* **Contextual Relevancy:** `0.625`


**📚 Citations:**

**[1]** `policies/RAG Security.txt (Chunk 7)`
> R9: Indirect Prompt Injection refers to a security risk where malicious prompts embedded within retrieved documents manipulate the generator to execut...

**[2]** `policies/RAG Security.txt (Chunk 7)`
> R9: Indirect Prompt Injection refers to a security risk where malicious prompts embedded within retrieved documents manipulate the generator to execut...

**[3]** `policies/RAG Security.txt (Chunk 4)`
> A. Background on RAG Systems To effectively analyze vulnerabilities and their mitigations in RAG Systems, it is essential to understand their architec...



### Another Example

In [8]:
from IPython.display import Markdown, display, clear_output
from src.search import query_with_evaluation
from src.index import get_vector_store
from src.embed import get_embeddings

embeddings = get_embeddings()
vector_store = get_vector_store(embeddings)
query = "How does the Bank of Montreal (BMO) structure its internal vector databases for retail banking customers?" 

result = query_with_evaluation(
    vector_store=vector_store,
    query=query,
    top_n=3,
    eval_threshold=0.5,
)

if result["passed"]:
    status_color = "#10b981"
    status_icon = "✅"
    status_text = "PASSED EVALUATION"
else:
    status_color = "#ef4444"
    status_icon = "🛑"
    status_text = "FAILED EVALUATION (Abstention Guardrail Triggered)"

metrics = result.get("metrics", {})
ans_rel = metrics.get('answer_relevancy', 0.0)
ctx_rel = metrics.get('contextual_relevancy', 0.0)

display_text = f"""
### 🔍 Query: *"{query}"*
<br>

**Evaluation Status:** <span style="color:{status_color}; font-weight:bold;">{status_icon} {status_text}</span>

**Agent Response:**
> {result["answer"]}

<br>

**DeepEval Metrics:**
* **Answer Relevancy:** `{ans_rel}`
* **Contextual Relevancy:** `{ctx_rel}`
"""

if not result["passed"] and result.get("feedback"):
    display_text += f"\n\n**Automated Feedback:**\n_{result['feedback']}_"

chunks = result.get("chunks", [])
if chunks:
    display_text += "\n\n**📚 Citations:**\n\n"
    for i, c in enumerate(chunks, 1):
        meta = c.get("metadata", {})
        source = meta.get("source_display", "unknown")
        preview = c.get("content", "")[:150].replace("\n", " ")
        display_text += f"**[{i}]** `{source}`\n> {preview}...\n\n"

clear_output(wait=True)
display(Markdown(display_text))


### 🔍 Query: *"How does the Bank of Montreal (BMO) structure its internal vector databases for retail banking customers?"*
<br>

**Evaluation Status:** <span style="color:#ef4444; font-weight:bold;">🛑 FAILED EVALUATION (Abstention Guardrail Triggered)</span>

**Agent Response:**
> I do not have enough information to answer.

<br>

**DeepEval Metrics:**
* **Answer Relevancy:** `0.0`
* **Contextual Relevancy:** `0.0`


**Automated Feedback:**
_The retrieved information does not appear relevant to your question. Please try rephrasing your question, or consider adding additional documents to the knowledge base if you believe the answer should be available._

**📚 Citations:**

**[1]** `policies/CB NoFee MCard Benefits Guide_EN.pdf (Chunk 5)`
> CashBack Redemption Register your credit card at bmocashback.com and tell us  how and when you want to redeem cash back rewards! Redemption options:  ...

**[2]** `policies/CB NoFee MCard Benefits Guide_EN.pdf (Chunk 5)`
> CashBack Redemption Register your credit card at bmocashback.com and tell us  how and when you want to redeem cash back rewards! Redemption options:  ...

**[3]** `policies/CB NoFee MCard Benefits Guide_EN.pdf (Chunk 11)`
> Our Commitment to You  BMO Financial Group appreciates and values  the opportunity to assist you in meeting  your financial objectives today, and in t...



## 4. RAG StateGraph with DeepEval-Driven Dynamic Routing

The single-pass approach above either succeeds or fails — it never retries. The **StateGraph** (built with LangGraph) wraps the same retrieve-generate-evaluate loop in a state machine that can *react* to evaluation results and try again with adjusted parameters.

### Routing logic

```
START ➔ retrieve ➔ generate ➔ evaluate ──┬── passed ──────➔ success ➔ END
                                          │
                                          ├── contextual relevancy failed
                                          │   (retries left)
                                          │   ➔ rephrase_query ➔ retrieve ➔ ...
                                          │
                                          └── answer relevancy failed
                                              or retries exhausted
                                              ➔ feedback_node ➔ END
```

### What happens on retry

When contextual relevancy fails and retries remain, the `rephrase_query` node does three things:

1. **Rephrases the query** using the LLM, aiming for better retrieval terms.
2. **Increases `top_n`** based on how far the contextual relevancy score was from the threshold. A larger gap triggers a bigger increase (e.g., score 0.1 with threshold 0.5 gives +4 chunks), capped at `top_n_max` (default 15).
3. **Relaxes `eval_threshold`** by 0.05 per retry, floored at `eval_threshold_floor` (default 0.3) to prevent over-relaxation.

The original query is preserved in state so the output can show both the original and rephrased versions, along with how `top_n` and `eval_threshold` changed. This makes the graph's decision-making fully transparent.

In [9]:
from IPython.display import Markdown, display, clear_output
from src.search import query_with_graph
from src.index import get_vector_store
from src.embed import get_embeddings

# 1. Initialize dependencies
embeddings = get_embeddings()
vector_store = get_vector_store(embeddings)
query = "What metrics are recommended for evaluating the retrieval and generation performance of a RAG pipeline, and how can auto-evaluation frameworks help identify pipeline errors?"
initial_top_n = 3
initial_eval_threshold = 0.5

# 2. Execute the StateGraph query
result = query_with_graph(
    query=query,
    vector_store=vector_store,
    top_n=initial_top_n,
    eval_threshold=initial_eval_threshold,
    max_retries=3,
)

# 3. Format the output beautifully using Markdown and HTML
if result.get("passed"):
    status_color = "#10b981"
    status_icon = "✅"
    status_text = "PASSED: Safe for User"
else:
    status_color = "#f59e0b"
    status_icon = "⚠️"
    status_text = "FAILED: Intercepted by Guardrails"

path_formatted = " ➔ ".join([f"**`{node}`**" for node in result.get("path_taken", [])])

metrics = result.get("metrics", {})
ans_rel = metrics.get('answer_relevancy', 0.0)
ctx_rel = metrics.get('contextual_relevancy', 0.0)

# Build rephrased-query section (only shown when a rephrase occurred)
rephrase_section = ""
if result.get("rephrased_query"):
    rephrase_section = f"""
**🔄 Query Rephrase:**
| | Query |
|---|---|
| **Original** | {result.get('original_query', query)} |
| **Rephrased** | {result['rephrased_query']} |
"""

# Build dynamic-params section (only shown when values changed from initial)
params_section = ""
final_top_n = result.get("top_n", initial_top_n)
final_threshold = result.get("eval_threshold", initial_eval_threshold)
if final_top_n != initial_top_n or final_threshold != initial_eval_threshold:
    params_section = f"""
**⚙️ Dynamic Parameter Adjustments:**
| Parameter | Initial | Final |
|---|---|---|
| `top_n` | {initial_top_n} | {final_top_n} |
| `eval_threshold` | {initial_eval_threshold:.2f} | {final_threshold:.2f} |
"""

display_text = f"""
### 🕸️ StateGraph Execution Trace
<br>

**🔍 Original Query:** *"{query}"*

**🚦 Final State:** <span style="color:{status_color}; font-weight:bold;">{status_icon} {status_text}</span>

**🛣️ Path Taken (Routing Logic):**
> {path_formatted}
{rephrase_section}{params_section}
**🤖 Agent Final Response:**
> {result.get("answer", "No answer generated.")}

<br>

**📊 DeepEval Guardrail Metrics:**
* **Contextual Relevancy (Docs match Query?):** `{ctx_rel:.2f}`
* **Answer Relevancy (Answer matches Query?):** `{ans_rel:.2f}`
"""

if result.get("feedback"):
    display_text += f"\n\n**🔄 Graph Feedback Node Output:**\n_{result['feedback']}_"

chunks = result.get("chunks", [])
if chunks:
    display_text += "\n\n**📚 Citations:**\n\n"
    for i, c in enumerate(chunks, 1):
        meta = c.get("metadata", {})
        source = meta.get("source_display", "unknown")
        preview = c.get("content", "")[:150].replace("\n", " ")
        display_text += f"**[{i}]** `{source}`\n> {preview}...\n\n"

clear_output(wait=True)
display(Markdown(display_text))


### 🕸️ StateGraph Execution Trace
<br>

**🔍 Original Query:** *"What metrics are recommended for evaluating the retrieval and generation performance of a RAG pipeline, and how can auto-evaluation frameworks help identify pipeline errors?"*

**🚦 Final State:** <span style="color:#10b981; font-weight:bold;">✅ PASSED: Safe for User</span>

**🛣️ Path Taken (Routing Logic):**
> **`retrieve`** ➔ **`generate`** ➔ **`evaluate`** ➔ **`success`**

**🤖 Agent Final Response:**
> The context does not provide specific metrics recommended for evaluating the retrieval and generation performance of a RAG pipeline. However, it mentions that simple QA benchmarks can be evaluated using exact match or overlap-based metrics, but realistic questions and answers require more sophisticated evaluations based on natural language understanding [3]. 

Regarding auto-evaluation frameworks, they are developed to classify error types according to a taxonomy, which helps practitioners identify weak links and common errors in their RAG pipelines. The auto-evaluation system can be used to identify the most common errors produced by a reference pipeline and implement targeted improvements [1], [3]. 

Thus, while specific metrics are not detailed, the auto-evaluation frameworks play a crucial role in error identification and mitigation in RAG systems.

<br>

**📊 DeepEval Guardrail Metrics:**
* **Contextual Relevancy (Docs match Query?):** `1.00`
* **Answer Relevancy (Answer matches Query?):** `1.00`


**📚 Citations:**

**[1]** `troubleshooting/Error Handling In RAG Systems.md (Chunk 2)`
> Existing work on RAG errors has generally not accounted for the complexity of real-world RAG systems and their failure modes. On the data side, widely...

**[2]** `troubleshooting/Error Handling In RAG Systems.md (Chunk 2)`
> Existing work on RAG errors has generally not accounted for the complexity of real-world RAG systems and their failure modes. On the data side, widely...

**[3]** `troubleshooting/Error Handling In RAG Systems.md (Chunk 3)`
> Our main contributions are: 1. A novel taxonomy of errors made by RAG systems, along with examples of each and recommendations for addressing them; 2....



### Another Example

In [10]:
from IPython.display import Markdown, display, clear_output
from src.search import query_with_graph
from src.index import get_vector_store
from src.embed import get_embeddings

embeddings = get_embeddings()
vector_store = get_vector_store(embeddings)
query = "How does the Bank of Montreal (BMO) structure its internal vector databases for retail banking customers?" 
initial_top_n = 3
initial_eval_threshold = 0.5

result = query_with_graph(
    query=query,
    vector_store=vector_store,
    top_n=initial_top_n,
    eval_threshold=initial_eval_threshold,
    max_retries=1,
)

if result.get("passed"):
    status_color = "#10b981"
    status_icon = "✅"
    status_text = "PASSED: Safe for User"
else:
    status_color = "#f59e0b"
    status_icon = "⚠️"
    status_text = "FAILED: Intercepted by Guardrails"

path_formatted = " ➔ ".join([f"**`{node}`**" for node in result.get("path_taken", [])])

metrics = result.get("metrics", {})
ans_rel = metrics.get('answer_relevancy', 0.0)
ctx_rel = metrics.get('contextual_relevancy', 0.0)

rephrase_section = ""
if result.get("rephrased_query"):
    rephrase_section = f"""
**🔄 Query Rephrase:**
| | Query |
|---|---|
| **Original** | {result.get('original_query', query)} |
| **Rephrased** | {result['rephrased_query']} |
"""

params_section = ""
final_top_n = result.get("top_n", initial_top_n)
final_threshold = result.get("eval_threshold", initial_eval_threshold)
if final_top_n != initial_top_n or final_threshold != initial_eval_threshold:
    params_section = f"""
**⚙️ Dynamic Parameter Adjustments:**
| Parameter | Initial | Final |
|---|---|---|
| `top_n` | {initial_top_n} | {final_top_n} |
| `eval_threshold` | {initial_eval_threshold:.2f} | {final_threshold:.2f} |
"""

display_text = f"""
### 🕸️ StateGraph Execution Trace
<br>

**🔍 Original Query:** *"{query}"*

**🚦 Final State:** <span style="color:{status_color}; font-weight:bold;">{status_icon} {status_text}</span>

**🛣️ Path Taken (Routing Logic):**
> {path_formatted}
{rephrase_section}{params_section}
**🤖 Agent Final Response:**
> {result.get("answer", "No answer generated.")}

<br>

**📊 DeepEval Guardrail Metrics:**
* **Contextual Relevancy (Docs match Query?):** `{ctx_rel:.2f}`
* **Answer Relevancy (Answer matches Query?):** `{ans_rel:.2f}`
"""

if result.get("feedback"):
    display_text += f"\n\n**🔄 Graph Feedback Node Output:**\n_{result['feedback']}_"

chunks = result.get("chunks", [])
if chunks:
    display_text += "\n\n**📚 Citations:**\n\n"
    for i, c in enumerate(chunks, 1):
        meta = c.get("metadata", {})
        source = meta.get("source_display", "unknown")
        preview = c.get("content", "")[:150].replace("\n", " ")
        display_text += f"**[{i}]** `{source}`\n> {preview}...\n\n"

clear_output(wait=True)
display(Markdown(display_text))


### 🕸️ StateGraph Execution Trace
<br>

**🔍 Original Query:** *"How does the Bank of Montreal (BMO) structure its internal vector databases for retail banking customers?"*

**🚦 Final State:** <span style="color:#f59e0b; font-weight:bold;">⚠️ FAILED: Intercepted by Guardrails</span>

**🛣️ Path Taken (Routing Logic):**
> **`retrieve`** ➔ **`generate`** ➔ **`evaluate`** ➔ **`rephrase_query`** ➔ **`retrieve`** ➔ **`generate`** ➔ **`evaluate`** ➔ **`feedback_node`**

**🔄 Query Rephrase:**
| | Query |
|---|---|
| **Original** | How does the Bank of Montreal (BMO) structure its internal vector databases for retail banking customers? |
| **Rephrased** | What is the structure of the internal vector databases used by the Bank of Montreal (BMO) for retail banking customers? |

**⚙️ Dynamic Parameter Adjustments:**
| Parameter | Initial | Final |
|---|---|---|
| `top_n` | 3 | 8 |
| `eval_threshold` | 0.50 | 0.45 |

**🤖 Agent Final Response:**
> The answer does not appear relevant to your question. Please try rephrasing your question for better results.

<br>

**📊 DeepEval Guardrail Metrics:**
* **Contextual Relevancy (Docs match Query?):** `0.00`
* **Answer Relevancy (Answer matches Query?):** `0.00`


**🔄 Graph Feedback Node Output:**
_The answer does not appear relevant to your question. Please try rephrasing your question for better results._

**📚 Citations:**

**[1]** `policies/CB NoFee MCard Benefits Guide_EN.pdf (Chunk 5)`
> CashBack Redemption Register your credit card at bmocashback.com and tell us  how and when you want to redeem cash back rewards! Redemption options:  ...

**[2]** `policies/CB NoFee MCard Benefits Guide_EN.pdf (Chunk 5)`
> CashBack Redemption Register your credit card at bmocashback.com and tell us  how and when you want to redeem cash back rewards! Redemption options:  ...

**[3]** `policies/bmo-eclipse-visa-infinite-benefit-guide-en.pdf (Chunk 13)`
> LEGAL 1  Primary Cardholders will be eligible to receive a $50 statement credit each year (annual period starts on the account anniversary date of acc...

**[4]** `policies/bmo-eclipse-visa-infinite-benefit-guide-en.pdf (Chunk 13)`
> LEGAL 1  Primary Cardholders will be eligible to receive a $50 statement credit each year (annual period starts on the account anniversary date of acc...

**[5]** `policies/CB NoFee MCard Benefits Guide_EN.pdf (Chunk 17)`
> 03/23-2144 Cut out the card below and keep it handy in case you need to reach us. BMO Credit Card Contact Information  Check your account online  bmo....

**[6]** `policies/CB NoFee MCard Benefits Guide_EN.pdf (Chunk 17)`
> 03/23-2144 Cut out the card below and keep it handy in case you need to reach us. BMO Credit Card Contact Information  Check your account online  bmo....

**[7]** `policies/CB NoFee MCard Benefits Guide_EN.pdf (Chunk 14)`
> A-Car partners must be made at least 24 hours in advance of scheduled  pick-up time. Additional mandatory charges may be imposed, including,  but not ...

**[8]** `policies/CB NoFee MCard Benefits Guide_EN.pdf (Chunk 14)`
> A-Car partners must be made at least 24 hours in advance of scheduled  pick-up time. Additional mandatory charges may be imposed, including,  but not ...

